# AI Agent — 판단과 실행을 자동으로 반복하기

앞 실습에서는 `step()`을 직접 호출하며 판단과 실행을 한 번씩 진행했습니다.
이번에는 `agent_runner.py`의 `run()`을 호출하여 이 과정을 자동으로 반복합니다.

실행 결과를 다음 판단에 전달하며 반복하는 구조를
**에이전트 루프(Agent Loop)**라고 합니다.

## run()을 호출하면

1. 현재 상태와 실행 이력을 초기화합니다.
2. `step()`을 호출하여 LLM의 판단과 도구 실행을 진행합니다.
3. 갱신된 상태와 실행 이력을 다음 `step()` 호출에 전달합니다.
4. LLM이 `action=None`을 반환하거나 최대 판단 횟수에 도달하면 반복을 멈춥니다.
5. 종료 상태와 이유, 최종 상태, 누적된 실행 이력을 노트북에 반환합니다.

## 종료 결과

- `SUCCESS`: 이용 가능한 교통편을 확인하고 LLM이 종료를 선택했습니다.
- `INCOMPLETE`: 교통편을 찾지 못한 상태에서 LLM이 종료를 선택했습니다.
- `MAX_STEPS`: LLM의 종료 선택 없이 최대 판단 횟수에 도달했습니다.

최대 판단 횟수에는 도구를 실행하지 않는 종료 판단도 포함됩니다.

## 실습 방법

`run()` 호출 셀을 실행하고, 출력되는 행동과 실행 결과를 확인하세요.
실행이 끝나면 최종 상태와 실행 이력을 확인합니다.

`run()`을 다시 호출하면 상태와 이력이 초기화되어 처음부터 실행됩니다.
이번 실습은 교통편 검색까지 진행하며 실제 예약이나 결제는 하지 않습니다.

In [1]:
import json

from agent_runner import run

result = run("다음 주 부산 출장 준비")

print("\n[최종 결과]")
print(json.dumps(result, ensure_ascii=False, indent=2))


[STEP 1]
선택한 행동: check_calendar
실행 인자: {'trip_date': None, 'meeting_time': None, 'transport_type': None}
선택 이유: 현재 상태에 출장 날짜와 미팅 시간이 없으므로 캘린더 확인이 필요합니다.
실행 결과: {'trip_date': '2026-10-08', 'meeting_time': '14:00'}

[STEP 2]
선택한 행동: search_transport
실행 인자: {'trip_date': '2026-10-08', 'meeting_time': '14:00', 'transport_type': 'KTX'}
선택 이유: 출장 날짜와 미팅 시간이 확인되었고 아직 교통편 검색 이력이 없으므로 KTX 좌석을 검색합니다.
실행 결과: {'transport_type': 'KTX', 'transport_status': 'sold_out', 'transport_message': 'KTX 좌석이 매진되었습니다.'}

[STEP 3]
선택한 행동: search_transport
실행 인자: {'trip_date': '2026-10-08', 'meeting_time': '14:00', 'transport_type': 'BUS'}
선택 이유: 출장 일정이 확인되었고 KTX가 매진되었으며 BUS 검색 이력은 없으므로 BUS 좌석을 검색합니다.
실행 결과: {'transport_type': 'BUS', 'transport_status': 'available', 'transport_message': '버스 좌석이 있습니다.'}

[STEP 4]
선택한 행동: None
실행 인자: {'trip_date': None, 'meeting_time': None, 'transport_type': None}
선택 이유: BUS 좌석을 이용할 수 있음이 확인되었으므로 교통편 검색을 종료합니다.

[최종 결과]
{
  "status": "SUCCESS",
  "reason": "최대 판단 횟수에 도달했습니다.",
  